# セッションセービスの拡張例

このノートブックでは、VertexAiSessionService を拡張して、セッション情報（会話履歴）に含まれる入出力メッセージのテキストを暗号化して保存する例を紹介します。

## 事前準備

**[SEE-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0 \
    google-cloud-kms==3.17.0

**[SEE-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform|google-cloud-kms)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-cloud-kms                      3.17.0
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-cloud-kms                         3.17.0
google-genai                             2.20.0
```

## ユーザー認証

**[SEE-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[SEE-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [2]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

事前に、クラウドコンソールのコマンド端末で以下を実行して、Cloud KMS にセッションの暗号鍵を保管しておきます。

```
PROJECT_ID=$(gcloud config list --format 'value(core.project)' 2>/dev/null)
PROJECT_NUMBER=$(gcloud projects describe $PROJECT_ID --format='value(projectNumber)' 2>/dev/null)
LOCATION='us-central1'

# キーリング 'session-keyring' を作成
gcloud kms keyrings create 'session-keyring' \
    --location=$LOCATION \
    --project=$PROJECT_ID

# 暗号鍵 'session-kek' を作成して、キーリング 'session-keyring' に保管
gcloud kms keys create 'session-kek' \
    --keyring='session-keyring' \
    --location=$LOCATION \
    --purpose='encryption' \
    --protection-level='software' \
    --project=$PROJECT_ID

# サービスアカウントに暗号鍵 'session-kek' へのアクセス権を設定
gcloud kms keys add-iam-policy-binding 'session-kek' \
    --keyring='session-keyring' \
    --location=$LOCATION \
    --member="serviceAccount:service-${PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com" \
    --role='roles/cloudkms.cryptoKeyEncrypterDecrypter' \
    --project=$PROJECT_ID
```

**[SEE-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

また、変数 `KMS_KEY_NAME` に作成した暗号化キーのリソース名を保存しておきます。

In [3]:
import base64, logging, os, secrets
from IPython.display import HTML, Markdown, display
from cryptography.fernet import Fernet
from google.cloud import kms
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk.agents.llm_agent import LlmAgent
from google.adk.sessions.vertex_ai_session_service import VertexAiSessionService
from google.adk.tools import google_search

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

key_ring = 'session-keyring'
key_name = 'session-kek'
KMS_KEY_NAME = f'projects/{PROJECT_ID}/locations/us-central1/keyRings/{key_ring}/cryptoKeys/{key_name}'

**[SEE-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

In [4]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
        return '\n'.join(result)

## 暗号化機能を追加した VertexAiSessionService の拡張クラスを作成

**[SEE-07]**

クラス `VertexAiSessionService` を拡張して、イベントのテキスト部分を暗号化する機能を追加したラッパークラス `MyVertexAiSessionService` を定義します。


In [5]:
logger = logging.getLogger(__name__)

class MyVertexAiSessionService(VertexAiSessionService):
    """
    エンベロープ暗号化（KEK として Cloud KMS、DEK としてローカル Fernet）を使用して、
    イベントのテキスト部分を暗号化する VertexAiSessionService のラッパー
    """

    def __init__(
        self, kms_key_name, project=None, location=None, agent_engine_id=None,
        *, express_mode_api_key=None,
    ):
        """
        MyVertexAiSessionService を初期化

        Args:
            kms_key_name: Cloud KMS CryptoKey の完全なリソース名
              - 例: "projects/my-project/locations/global/keyRings/my-ring/cryptoKeys/my-key"
            project: Agent Platform Sessions サービスのプロジェクトID
            location: Agent Platform Sessions サービスのロケーション
            agent_engine_id: 環境変数 "GOOGLE_CLOUD_AGENT_ENGINE_ID" から取得できる ID
            express_mode_api_key: Express Mode で使用する API キー
        """
        super().__init__(
            project=project,
            location=location,
            agent_engine_id=agent_engine_id,
            express_mode_api_key=express_mode_api_key,
        )
        self._kms_key_name = kms_key_name
        self._kms_client = kms.KeyManagementServiceClient()

    def _encrypt_text(self, plaintext):
        """エンベロープ暗号化を使用して平文文字列を暗号化"""
        # encypt plaintext with DEK.
        raw_dek = secrets.token_bytes(32)
        fernet = Fernet(base64.urlsafe_b64encode(raw_dek))
        ciphertext = fernet.encrypt(plaintext.encode('utf-8')).decode('utf-8')

        # encrypt raw_dek with KMS.
        kms_response = self._kms_client.encrypt(
            request={
                'name': self._kms_key_name,
                'plaintext': raw_dek,
            }
        )
        encrypted_dek_b64 = base64.urlsafe_b64encode(kms_response.ciphertext).decode('utf-8')

        # return "<encrypted_raw_dek>:<encrypted plaintext>".
        return f"{encrypted_dek_b64}:{ciphertext}"

    def _decrypt_text(self, envelope_payload):
        """エンベロープ暗号化された文字列を復号"""
        if ':' not in envelope_payload:
            raise ValueError('Invalid envelope payload format: missing delimiter.')
        encrypted_dek_b64, ciphertext = envelope_payload.split(':', 1)

        # decrypt raw_dek with KMS.
        encrypted_dek = base64.urlsafe_b64decode(encrypted_dek_b64)
        kms_response = self._kms_client.decrypt(
            request={
                'name': self._kms_key_name,
                'ciphertext': encrypted_dek,
            }
        )
        raw_dek = kms_response.plaintext

        # decrypt plaintext with DEK.
        fernet = Fernet(base64.urlsafe_b64encode(raw_dek))
        decrypted_text = fernet.decrypt(ciphertext.encode('utf-8')).decode('utf-8')

        # return decrypted plaintext.
        return decrypted_text

    def _process_event(self, event, encrypt):
        """イベント内のテキスト部分をインプレースで暗号化または復号"""
        if not event.content or not event.content.parts:
            return
        for part in event.content.parts:
            if part.text:
                try:
                    if encrypt:
                        part.text = self._encrypt_text(part.text)
                    else:
                        part.text = self._decrypt_text(part.text)
                except Exception as e:
                    action = 'encrypt' if encrypt else 'decrypt'
                    logger.warning(
                        'Failed to %s text part in event %s: %s',
                        action, event.id, e,
                    )

    # override
    async def append_event(self, session, event):
        """テキスト部分を暗号化した後、セッションにイベントを追加"""
        encrypted_event = event.model_copy(deep=True)
        self._process_event(encrypted_event, encrypt=True)
        await super().append_event(session, encrypted_event)
        # メモリ上のイベントは非暗号化状態を維持
        if session.events and session.events[-1] is encrypted_event:
            session.events[-1] = event
        return event

    # override
    async def get_session(
        self, *, app_name, user_id, session_id, config=None,
    ):
        """セッションを取得し、そのイベントのテキスト部分を復号"""
        session = await super().get_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_id,
            config=config,
        )
        if session:
            for event in session.events:
                self._process_event(event, encrypt=False)
        return session

## LlmAgent オブジェクトと AdkApp オブジェクトの作成

**[SEE-08]**

Grounding with Google Search を利用して、ユーザーの質問に回答する AI エージェント（LlmAgent オブジェクト）を作成します。

これを含む AdkApp オブジェクトを定義する際に、`session_service_builder` オプションで、クラス `MyVertexAiSessionService` の SessionService オブジェクトを使用するように設定します。

In [7]:
instruction = '''
あなたはユーザーの質問に回答するエージェントです。
- google_search を使用して、最新情報に基づいて回答してください。
- フレンドリーな会話を心がけてください。
'''

search_agent = LlmAgent(
    name='search_agent',
    model='gemini-3.5-flash-lite',
    description='Google 検索を用いて質問に回答するエージェント',
    instruction=instruction,
    tools=[google_search],
)

def session_builder():
    return MyVertexAiSessionService(
        kms_key_name=KMS_KEY_NAME,
        location='us-central1',
        agent_engine_id=os.environ.get('GOOGLE_CLOUD_AGENT_ENGINE_ID'),
    )

search_agent_app = AdkApp(
    agent=search_agent,
    app_name='search_agent_app',
    session_service_builder=session_builder,
)

## Agent Runtime へのデプロイ

**[SEE-09]**

用意した AdkApp オブジェクトを Agent Runtime にデプロイします。

In [16]:
agent_runtime = agentplatform.Client(location='us-central1').runtimes

display_name = 'Encrypted Search Agent App'

requirements = [
    'google-adk==2.8.0',
    'google-cloud-aiplatform==2.0.1',
    'google-cloud-kms==3.17.0',
    'google-genai==2.20.0',
]

config={
    'agent_framework': 'google-adk',
    'requirements': requirements,
    'staging_bucket': f'gs://{PROJECT_ID}_search_agent_app',
    'display_name': display_name,
    'env_vars': {
        'GOOGLE_CLOUD_LOCATION': 'global',
        'GOOGLE_GENAI_USE_VERTEXAI': 'True',
    },
}

remote_adk_app = agent_runtime.create(
    agent=search_agent_app,
    config=config,
)

INFO:agentplatform_genai.runtimes:View progress and logs at https://console.cloud.google.com/logs/query?project=etsuji-15pro-poc&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%225664265542127583232%22.
INFO:agentplatform_genai.runtimes:Agent Runtime created. To use it in another session:
INFO:agentplatform_genai.runtimes:runtime=client.runtimes.get(name='projects/848570887571/locations/us-central1/reasoningEngines/5664265542127583232')


デプロイ完了まで数分かかります。デプロイの進捗状況は、出力メッセージにある Cloud Logging へのリンクから確認できます。

```
INFO:agentplatform_genai.runtimes:View progress and logs at https://console.cloud.google.com/logs/query?project=...
```

## 利用例

**[SEE-10]**

デプロイしたAIエージェントと会話します。

In [19]:
chat_client = ChatClient(remote_adk_app)

query = '''
樹木希林の生年月日を教えてください。
'''
response = await chat_client.async_stream_query(query)
display(Markdown(response))

樹木希林さんの生年月日は、**1943年（昭和18年）1月15日**です。

**[SEE-11]**

会話履歴を正しく認識していることを確認します。

In [20]:
query = '''
誰の話をしていたか思い出してください。
'''
response = await chat_client.async_stream_query(query)
display(Markdown(response))

先ほどは、名優として知られる**樹木希林**さんのお話しをしていましたよ！

**[SEE-12]**

通常の VertexAiSessionService では、会話履歴が見れないことを確認します。

はじめにデプロイした AI エージェントのリソース名を取得します。

In [21]:
display_name = 'Encrypted Search Agent App'

for item in agent_runtime.list():
    resource = item.api_resource
    if resource.display_name == display_name:
        resource_name = resource.name
        break

print(f'Resource Name for {display_name}: {resource_name}')

Resource Name for Encrypted Search Agent App: projects/848570887571/locations/us-central1/reasoningEngines/5664265542127583232


**[SEE-13]**

ユーザー ID `defaulut_user` のセッション一覧を取得します。

In [22]:
session_service = VertexAiSessionService(location='us-central1')

session_list = await session_service.list_sessions(
    app_name=resource_name,
    user_id='default_user',
)
session_list = session_list.model_dump()
session_list

{'sessions': [{'id': '3117478612819771392',
   'app_name': 'projects/848570887571/locations/us-central1/reasoningEngines/5664265542127583232',
   'user_id': 'default_user',
   'state': {},
   'events': [],
   'last_update_time': 1789814166.776334}]}

**[SEE-14]**

最初のセッションの内容を取得して、最初のイベントの内容を表示します。

`text` 要素が暗号化されていて、実際の会話の内容は読み取れません。

**注意**: MyVertexAiSessionService を使用すると、会話の内容が復号されて読み取れますが、これは、鍵へのアクセス権を持ったユーザーしか利用できません。今の場合、プロジェクトオーナーのような強力な管理権限を持つユーザー以外では、Agent Runtime で稼動する AI エージェントを実行する Vertex AI Reasoning Engine Service Agent に限られます。

In [23]:
session_id = session_list['sessions'][0]['id']

session = await session_service.get_session(
    app_name=resource_name,
    user_id='default_user',
    session_id=session_id
)
session = session.model_dump()
session['events'][0]['content']

{'parts': [{'media_resolution': None,
   'code_execution_result': None,
   'executable_code': None,
   'file_data': None,
   'function_call': None,
   'function_response': None,
   'inline_data': None,
   'text': 'CiQAPFOEc4Vr_-etF-tA0U-IiCUa-VlMHwONqF_s78s3SNByG5YSSQCwOoNn-UTjUfJN1-ts0IAZJiDcVAI8ZSFUGSWfD-YTeJta1BuqGfauFbgaI_IbQkgF0Dvv-ONTZsGXQVu44GA6KTlZnPCxn84=:gAAAAABqrmWTuiI3KmWShnXAqq5d4zAY5tyTSPbr9Hr7KC6iRREc2X46X13Ss5oWoP9gbTTfNom-VGVSxmoLRGEwIqurVpk5VRocXFJLl1Jg92xHrNHmtEQdhEIA8D2PQiUyHHZCoi1xI22txKHOureccXbLxW1GBg==',
   'thought': None,
   'thought_signature': None,
   'video_metadata': None,
   'tool_call': None,
   'tool_response': None,
   'part_metadata': None,
   'audio_transcription': None,
   'media_processing': None}],
 'role': 'user'}